In [11]:
!pip3 install opendatasets



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os

In [13]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/andrewmvd/animal-faces")

Skipping, found downloaded files in ".\animal-faces" (use force=True to force download)


In [ ]:
for i in os.listdir("fourth_project\animal-faces"):
    for label in os.listdir(f"second_project/animal-faces/{i}"):
        label_path = f"second_project/animal-faces/{i}/{label}"
        for image in os.listdir(label_path):
            print(f"Label: {label}, Image: {image}")
            image_path.append(os.path.join(label_path, image))
            labels.append(label)
            break

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'second_project/animal-faces'

In [ ]:
# image_path=[]
# labels=[]
# for i in os.listdir(r"second_project\animal-faces"):
#     for label in os.listdir(f"second_project\animal-faces/{i}"):
#         for image in os.listdir(f"second_project\animal-faces/{i},{label}"):
#             print(label)
#             for image in os.listdir(f"second_project\animal-faces/{i}/{label}"):
#                 print(image)
#                 break
#             break

In [ ]:
!pip3 install torch
!pip3 install scikit-learn
!pip3 install pandas
!pip3 install torchvision


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import torch
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import pandas as pd
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import os
import numpy as np
from torchvision import models
from torchvision.transforms import transforms
# import matplotlib.pyplot as plt
device="cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
image_path=[]
labels=[]


In [ ]:
data_df=pd.DataFrame(zip(image_path,labels),columns=["image_path","labels"])
print(data_df["labels"].unique())
data_df.head()

[]


,image_path,labels


In [ ]:
train_df=pd.read_csv(r'second_project\train.csv')
val_df=pd.read_csv(r'second_project\val.csv')
# train_df["image:FILE"]=

FileNotFoundError: [Errno 2] No such file or directory: 'second_project\\train.csv'

In [ ]:
train_df.head()

NameError: name 'train_df' is not defined

In [ ]:
print(train_df.shape)

(1034, 2)


In [ ]:
# label_encoder=LabelEncoder()
# label_encoder.fit(train_df["labels"])


In [ ]:
## Transform Function

In [ ]:
transform=transforms.Compose([transforms.Resize((128,128)),transforms.ToTensor(),transforms.ConvertImageDtype(torch.float)])

In [ ]:
class CustomImageDataset(Dataset):
    def __init__(self,dataframe,transform):
        self.dataframe=dataframe
        self.transform=transform
        self.labels=torch.tensor(dataframe["category"]).to(device)
    def __len__(self):
        return self.dataframe.shape[0]
    def __getitem__(self, idx):
        img_path = self.dataframe.iloc[idx, 0]
        label = self.labels[idx]  # ✅ correct indexing
        image = Image.open(img_path)

        if self.transform:
            image = self.transform(image)

        return image, label

        

In [ ]:
train_dataset=CustomImageDataset(dataframe=train_df,transform=transform)
val_dataset=CustomImageDataset(dataframe=val_df,transform=transform)
# train_df["image:FILE"]=+train_df["image:FILE"]
# train_df["image:FILE"]=+train_df["image:FILE"]

In [ ]:
# train_dataset.__getitem__(2)

In [ ]:
# n_rows = 3
# n_cols = 3
# f, axarr = plt.subplots(n_rows, n_cols, figsize=(10, 10))

# for row in range(n_rows):
#     for col in range(n_cols):
#         idx = row * n_cols + col
#         image, label = train_dataset[idx]  # ✅ get image from dataset
#         image = image.permute(1, 2, 0).numpy()  # CHW → HWC and convert to NumPy
#         axarr[row, col].imshow(image)
#         axarr[row, col].set_title(f"Label: {label}")
#         axarr[row, col].axis('off')

# plt.tight_layout()
# plt.show()


In [ ]:
lr=1e-3
BATCH_SIZE=4
EPOCHS=15

In [ ]:
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE,shuffle=True)
googlenet_model=models.googlenet(weights='DEFAULT')

In [ ]:
train_loader

In [ ]:
class Net(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3)
        self.pooling = nn.MaxPool2d(2, 2)
        self.relu = nn.ReLU()
        self.flatten = nn.Flatten()
        self.linear = nn.Linear(128*14*14, 128)  # corrected size
        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.relu(self.pooling(self.conv1(x)))
        x = self.relu(self.pooling(self.conv2(x)))
        x = self.relu(self.pooling(self.conv3(x)))
        x = self.flatten(x)
        x = self.linear(x)
        x = self.output(x)
        return x

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=Adam(model.parameters(),lr=lr)
num_classes = len(train_df["category"].unique())
model = Net(num_classes).to(device)

In [ ]:
from torchsummary import summary
summary(model,input_size=(3,128,128))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 126, 126]             896
         MaxPool2d-2           [-1, 32, 63, 63]               0
              ReLU-3           [-1, 32, 63, 63]               0
            Conv2d-4           [-1, 64, 61, 61]          18,496
         MaxPool2d-5           [-1, 64, 30, 30]               0
              ReLU-6           [-1, 64, 30, 30]               0
            Conv2d-7          [-1, 128, 28, 28]          73,856
         MaxPool2d-8          [-1, 128, 14, 14]               0
              ReLU-9          [-1, 128, 14, 14]               0
          Flatten-10                [-1, 25088]               0
           Linear-11                  [-1, 128]       3,211,392
           Linear-12                    [-1, 3]             387
Total params: 3,305,027
Trainable params: 3,305,027
Non-trainable params: 0
---------------------------

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=lr)

total_loss_train_plot = []
total_loss_validation_plot = []
total_acc_train_plot = []
total_acc_validation_plot = []

for epoch in range(EPOCHS):
    total_acc_train = 0
    total_loss_train = 0
    total_loss_val = 0
    total_acc_val = 0

    model.train()
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)  
        train_loss = criterion(outputs, labels)
        train_loss.backward()
        optimizer.step()

        total_loss_train += train_loss.item()
        train_acc = (torch.argmax(outputs, axis=1) == labels).sum().item()
        total_acc_train += train_acc

    model.eval()
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            val_loss = criterion(outputs, labels)
            total_loss_val += val_loss.item()
            val_acc = (torch.argmax(outputs, axis=1) == labels).sum().item()
            total_acc_val += val_acc

    # Append metrics after each epoch
    total_loss_train_plot.append(round(total_loss_train / len(train_loader), 4))
    total_loss_validation_plot.append(round(total_loss_val / len(val_loader), 4))
    total_acc_train_plot.append(round(total_acc_train / len(train_dataset) * 100, 4))
    total_acc_validation_plot.append(round(total_acc_val / len(val_dataset) * 100, 4))

    # Print epoch summary
    print(f"Epoch {epoch+1}, "
          f"Train Loss: {total_loss_train_plot[-1]} "
          f"Train Accuracy: {total_acc_train_plot[-1]}% "
          f"Validation Loss: {total_loss_validation_plot[-1]} "
          f"Validation Accuracy: {total_acc_validation_plot[-1]}%")


FileNotFoundError: [Errno 2] No such file or directory: 'train/bean_rust/bean_rust_train.115.jpg'

In [ ]:
with torch.no_grad():
    total_loss_test=0
    total_acc_test=0
    for input,labels in val_loader:
        prediction=model(input)
        acc=(torch.argmax(prediction,axis=1)==labels).sum().item()
        total_acc_test+=acc
        total_loss=criterion(prediction,labels)
        total_loss_test+=total_loss.item()
    print(f"Accuracy score is: {round(total_acc_test/test_data.__len__())*100,4} and loss is {round(total_loss_test/1000,4)}")

FileNotFoundError: [Errno 2] No such file or directory: 'val/angular_leaf_spot/angular_leaf_spot_val.14.jpg'

In [ ]:
fig,axs=plt.subplot(n_rows=1,n_cols=2,figsize=(15,5))
axs[0].plot(total_loss_train_plot,label='Training loss')
axs[0].set_title("Training and validation loss over epochs")
axs[0].set_xlabel("Epochs")
axs[0].set_ylabel("loss")
axs.legend()


In [ ]:
##1-read image
##2- Transform using Transform
##3- predict throguh model
##4- Inverse transform by label encoder


In [ ]:
def predict_image(image_path):
    image=Image.open(image_path).covnert("RGB")
    image=transform(image).to(device)
    output=model(image.unsqueeze(0))
    return output

predict_image("path")
